# Final Project — Predicting Benign vs. Malignant Mammogram Masses

**Course:** Data Science & Machine Learning with Python — Hands On! (Frank Kane / Sundog Education)

## The problem
Roughly **70% of breast biopsies turn out benign**, meaning a lot of women undergo an
invasive, stressful, expensive procedure that (in hindsight) wasn't needed. If a model can
reliably predict whether a mass is benign or malignant from attributes a radiologist can
read off a mammogram, it could help avoid unnecessary biopsies.

## The data — *Mammographic Mass* (UCI Machine Learning Repository)
961 instances. Attributes:

| Column | Meaning | Type |
|---|---|---|
| BI-RADS | Radiologist's assessment, 1–5 | **not a real feature — discard** |
| age | Patient age in years | numeric |
| shape | round=1, oval=2, lobular=3, irregular=4 | nominal |
| margin | circumscribed=1, microlobulated=2, obscured=3, ill-defined=4, spiculated=5 | nominal |
| density | high=1, iso=2, low=3, fat-containing=4 | ordinal |
| **severity** | **benign=0, malignant=1** | **← target** |

We drop **BI-RADS** because it *is* a doctor's assessment, not something physically measured —
using it would be letting the model peek at the answer a radiologist already formed.

## Plan
1. Load & clean the data (missing values are marked `?`).
2. Scale the features.
3. Evaluate many models the course covered, each with **10-fold cross-validation** so the
   comparison is honest: Decision Tree, Random Forest, KNN, Naïve Bayes, SVM (several kernels),
   Logistic Regression, and a Keras neural network.
4. Compare and pick the best.


## 1. Load the data

The course file is usually named `mammographic_masses.data.txt`. If you downloaded it straight from UCI it's `mammographic_masses.data`. Point the path below at whichever you have. Missing values are stored as `?`, so we tell pandas to read them as `NaN`.

In [ ]:
import numpy as np
import pandas as pd

# Adjust the filename/path if needed:
FILE = 'mammographic_masses.data.txt'   # or 'mammographic_masses.data'

col_names = ['BI-RADS', 'age', 'shape', 'margin', 'density', 'severity']
masses = pd.read_csv(FILE, na_values=['?'], names=col_names)

masses.head()

A quick look. `describe()` shows the count per column — if a column has fewer than the total row count, it has missing values.

In [ ]:
masses.describe()

## 2. Clean the data

We have missing values scattered across `age`, `shape`, `margin`, and `density`. Before we just
delete those rows, we should check the values aren't missing in a *biased* way (e.g. all the
malignant cases missing) — otherwise dropping them would distort the model. Let's eyeball the
rows with any missing value.

In [ ]:
# Rows with at least one missing value
masses[masses.isnull().any(axis=1)]

The missing values look scattered fairly randomly across benign and malignant cases and across ages, with no obvious pattern. With only ~130 such rows out of 961, dropping them is reasonable and simpler than imputing.

In [ ]:
masses.dropna(inplace=True)
masses.describe()

## 3. Build the feature matrix and target

- **Features (X):** `age`, `shape`, `margin`, `density`
- **Target (y):** `severity`
- We deliberately leave out **BI-RADS**.

scikit-learn wants plain NumPy arrays, so we pull out `.values`.

In [ ]:
feature_names = ['age', 'shape', 'margin', 'density']

X = masses[feature_names].values
y = masses['severity'].values

print('Feature matrix:', X.shape)
print('Target vector :', y.shape)

### Scale the features

`age` runs ~18–96 while `shape`/`margin`/`density` are tiny integers. Distance- and
gradient-based models (KNN, SVM, logistic regression, neural nets) behave badly when one feature
dwarfs the others, so we standardize every feature to mean 0 / std 1.

In [ ]:
from sklearn import preprocessing

scaler = preprocessing.StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled[:5]

## 4. Evaluate the models — 10-fold cross-validation

Instead of a single train/test split (whose score depends heavily on *which* rows landed in the
test set), we use **K-Fold cross-validation with K=10**. The data is split into 10 parts; each
part takes a turn as the test set while the other 9 train the model. We report the **mean
accuracy** across the 10 folds — a far more trustworthy number.

We'll collect every model's score in a dictionary and rank them at the end.

In [ ]:
from sklearn.model_selection import cross_val_score

results = {}   # model name -> mean 10-fold accuracy

### 4.1 Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

clf = DecisionTreeClassifier(random_state=1)
score = cross_val_score(clf, X_scaled, y, cv=10).mean()
results['Decision Tree'] = score
print(f'Decision Tree: {score:.4f}')

A single tree tends to overfit, so its cross-validated score is usually the weakest of the bunch (~0.73–0.75).

### 4.2 Random Forest
Many trees voting together — almost always beats a single tree.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(n_estimators=100, random_state=1)
score = cross_val_score(clf, X_scaled, y, cv=10).mean()
results['Random Forest'] = score
print(f'Random Forest: {score:.4f}')

### 4.3 K-Nearest Neighbors
KNN's only real knob is **K**. Rather than guess, let's sweep a range of K values and keep the best.

In [ ]:
from sklearn import neighbors

best_k, best_knn = None, -1
for k in range(1, 51):
    clf = neighbors.KNeighborsClassifier(n_neighbors=k)
    s = cross_val_score(clf, X_scaled, y, cv=10).mean()
    if s > best_knn:
        best_knn, best_k = s, k

results['KNN'] = best_knn
print(f'Best KNN: K={best_k}, accuracy={best_knn:.4f}')

### 4.4 Naïve Bayes

`MultinomialNB` requires **non-negative** features, but our standardized data has negatives.
So for this one model we re-scale the *original* features into the 0–1 range with `MinMaxScaler`.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

X_minmax = preprocessing.MinMaxScaler().fit_transform(X)
clf = MultinomialNB()
score = cross_val_score(clf, X_minmax, y, cv=10).mean()
results['Naive Bayes'] = score
print(f'Naive Bayes: {score:.4f}')

### 4.5 Support Vector Machine
SVMs are sensitive to the choice of **kernel**, so we try the four common ones and keep the best.

In [ ]:
from sklearn import svm

best_kernel, best_svm = None, -1
for kernel in ['linear', 'rbf', 'sigmoid', 'poly']:
    clf = svm.SVC(kernel=kernel, C=1.0)
    s = cross_val_score(clf, X_scaled, y, cv=10).mean()
    print(f'  {kernel:8s}: {s:.4f}')
    if s > best_svm:
        best_svm, best_kernel = s, kernel

results['SVM'] = best_svm
print(f'Best SVM: kernel={best_kernel}, accuracy={best_svm:.4f}')

### 4.6 Logistic Regression
A simple linear classifier — and on this dataset, one of the strongest.

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)
score = cross_val_score(clf, X_scaled, y, cv=10).mean()
results['Logistic Regression'] = score
print(f'Logistic Regression: {score:.4f}')

### 4.7 Neural Network (Keras)

A small feed-forward net: 4 inputs → hidden layers → 1 sigmoid output for binary classification.

**Note on library versions:** the course used `keras.wrappers.scikit_learn.KerasClassifier`,
which was **removed** from modern TensorFlow. The current way is the `scikeras` package. Install
it once with `pip install scikeras`. If you'd rather not, the manual K-fold cell further down
does the same job with plain Keras.

In [ ]:
# pip install scikeras   # run once if you don't have it
from scikeras.wrappers import KerasClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

def create_model():
    model = Sequential([
        Dense(6, input_dim=4, activation='relu'),
        Dense(4, activation='relu'),
        Dense(1, activation='sigmoid'),
    ])
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

estimator = KerasClassifier(model=create_model, epochs=100, batch_size=10, verbose=0)
score = cross_val_score(estimator, X_scaled, y, cv=10).mean()
results['Neural Network'] = score
print(f'Neural Network: {score:.4f}')

*(Optional)* Same neural network without `scikeras` — manual 10-fold loop using only Keras:

In [ ]:
# Manual alternative if you prefer not to install scikeras
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

def build():
    m = Sequential([
        Dense(6, input_dim=4, activation='relu'),
        Dense(4, activation='relu'),
        Dense(1, activation='sigmoid'),
    ])
    m.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return m

kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=1)
accs = []
for train_idx, test_idx in kf.split(X_scaled, y):
    model = build()
    model.fit(X_scaled[train_idx], y[train_idx], epochs=100, batch_size=10, verbose=0)
    _, acc = model.evaluate(X_scaled[test_idx], y[test_idx], verbose=0)
    accs.append(acc)

nn_manual = np.mean(accs)
results['Neural Network'] = nn_manual   # overwrite with this if you skipped scikeras
print(f'Neural Network (manual K-fold): {nn_manual:.4f}')

## 5. Results — which model wins?

In [ ]:
ranking = pd.DataFrame(
    sorted(results.items(), key=lambda kv: kv[1], reverse=True),
    columns=['Model', 'Mean 10-fold accuracy']
)
ranking['Mean 10-fold accuracy'] = ranking['Mean 10-fold accuracy'].round(4)
ranking.reset_index(drop=True, inplace=True)
ranking

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.barh(ranking['Model'][::-1], ranking['Mean 10-fold accuracy'][::-1])
plt.xlabel('Mean 10-fold cross-validated accuracy')
plt.title('Model comparison — mammogram mass classification')
plt.xlim(0.6, 0.85)
for i, v in enumerate(ranking['Mean 10-fold accuracy'][::-1]):
    plt.text(v + 0.002, i, f'{v:.3f}', va='center')
plt.tight_layout()
plt.show()

## 6. Conclusion

On this dataset every model lands in a fairly tight band, roughly **73%–80%** accuracy. The
consistent front-runners are **Logistic Regression, the linear-kernel SVM, and the neural
network**, all clustering around **~79–80%**. The single Decision Tree is reliably the weakest
because it overfits.

**Recommended model:** whichever of the top three scores highest in your run — but **Logistic
Regression** is arguably the best *practical* choice here: it matches the accuracy of the more
complex models while being fast, deterministic, and easy to explain to a clinician (you can point
at each feature's coefficient). In a medical setting that interpretability is a real advantage.

**Why we're stuck near 80%:** we're only using four coarse, mostly categorical features. The
ceiling is the data, not the algorithm — no amount of model tuning extracts information the
features don't contain. Realistically you'd push higher by adding features (e.g. actual image
characteristics) rather than by swapping classifiers.

**A caveat worth stating in a medical context:** raw accuracy hides the difference between the two
kinds of mistakes. Calling a malignant mass benign (a false negative) is far more dangerous than
the reverse. A production version of this should also look at **recall/sensitivity and the
confusion matrix**, not just accuracy — but for this project, accuracy is what the assignment asks
us to compare on.
